In [ ]:
from dotenv import load_dotenv
import os


load_dotenv()



OPEN_ROUTER_API_KEY = os.getenv("OPEN_ROUTER_API_KEY")
# OPEN_ROUTER_COMPLETION_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"

OPEN_ROUTER_COMPLETION_MODEL = "qwen3:latest"


OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
OLLAMA_API_KEY='ollama'


In [ ]:
import openai
from openai import OpenAI

# client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=OPEN_ROUTER_API_KEY,
#     default_headers={
#         "HTTP-Referer": "https://github.com/pixelbuildlab/be-story-teller",
#         "X-OpenRouter-Title": "Be story teller - Agentic way to stories",
#     },
# )

print(OLLAMA_API_URL)
client = OpenAI(
    base_url=OLLAMA_API_URL,
    api_key=OLLAMA_API_KEY,
)

In [ ]:
import json

In [ ]:
from typing import Optional

In [ ]:
SYSTEM_META_PROMPT="""
You are a professional children's story writer.

Your goals:
- Write bedtime stories for children aged 4–9.
- Stories should be calming.
- Never include violence or horror.
- Keep language simple.
- If you need to use a tool, use it.
- Stories should be relaxing, pleasing to hear and lesson full.
- Note: Story should be only of 20 characters for now.
"""

In [ ]:
def AI(messages: list, tools: Optional[list] | None):
    chat_completion = client.chat.completions.create(
        model=OPEN_ROUTER_COMPLETION_MODEL, messages=messages, tools=tools
    )
    return chat_completion

In [ ]:
OPEN_ROUTER_COMPLETION_MODEL

In [ ]:
META_PROMPT_OPTIMIZER = """
You are an expert prompt optimizer for children's story generation.

Your task is to transform short or incomplete user requests into rich, detailed prompts for a story-writing AI.

Rules:
- Preserve the user's original intent.
- Add reasonable assumptions when details are missing.
- Specify:
  - protagonist
  - setting
  - conflict
  - tone
  - target age
  - ending
  - approximate length
- Do not write the story.
- Return ONLY the optimized prompt.
"""


async def StoryPromptOptimizer(prompt: str):
    print("starting StoryPromptOptimizer")
    messages = [
        {"role": "system", "content": META_PROMPT_OPTIMIZER},
        {
            "role": "user",
            "content": f"{prompt}",
        },
    ]
    
    chat_completion = AI(messages, None)

    response_message = chat_completion.choices[0].message

    print("StoryPromptOptimizer called")
    return response_message, chat_completion

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "StoryPromptOptimizer",
            "description": "Optimize user input prompt to a level it creates stunning storyline",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                        "description": "User input prompt to optimize",
                    }
                },
                "required": ["prompt"],
            },
        },
    }
]

In [ ]:
MESSAGES_LIST = []

In [ ]:
async def openai_chat_completion(prompt: str, META_PROMPT: str):
    try:
        messages = [
            {"role": "system", "content": META_PROMPT},
            {
                "role": "user",
                "content": f"{prompt}",
            },
        ]
        MESSAGES_LIST.extend(messages)

        while True:
            print("STARTING AGENT")
            chat_completion = AI(MESSAGES_LIST, tools)
            response_message = chat_completion.choices[0].message

            MESSAGES_LIST.append(response_message.model_dump())
            print(f"MAIN chat output: {response_message}")

            # If LLM returned tool calls, process them
            if hasattr(response_message, "tool_calls") and response_message.tool_calls:
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    print(f"Tool call: {function_name}, args: {function_args}")
                    agent_args = []

                    tool_function = globals()[function_name]
                    
                    tool_result, tool_chat_completion = await tool_function(
                        *agent_args, **function_args
                    )

                    MESSAGES_LIST.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "content": tool_result.content,
                        }
                    )

            else:
                return chat_completion

            # return chat_completion

    except openai.APIConnectionError as e:
        print(f"Network connectivity issue: {e}")
    except openai.RateLimitError as e:
        print(f"Rate limits hit or out of funds: {e}")
    except openai.APIStatusError as e:
        print(f"HTTP Error received (Status: {e.status_code}): {e.response}")

In [ ]:
await openai_chat_completion('rabbit', SYSTEM_META_PROMPT)

In [ ]:
MESSAGES_LIST

In [ ]:
MESSAGES_LIST[-1]['content']

In [ ]:
# ChatCompletion(
#     id="gen-1784740420-a3TdQhM44Qqj99iyUHf7",
#     # choices=[
#     #     Choice(
#     #         finish_reason="tool_calls",
#     #         index=0,
#     #         logprobs=None,
#     #         message=ChatCompletionMessage(
#     #             content=None,
#     #             refusal=None,
#     #             role="assistant",
#     #             annotations=None,
#     #             audio=None,
#     #             function_call=None,
#     #             tool_calls=[
#     #                 ChatCompletionMessageFunctionToolCall(
#     #                     id="call-251336db-fc7d-4afb-a457-63afc788d290",
#     #                     function=Function(
#     #                         arguments='{"prompt":"rabbit"}', name="PromptOptimizerTool"
#     #                     ),
#     #                     type="function",
#     #                     index=0,
#     #                 )
#     #             ],
#     #             reasoning="The user wants a bedtime story about a rabbit. I should use the PromptOptimizerTool to optimize this simple prompt into a better storyline before writing the story. Let me do that first.",
#     #             reasoning_details=[
#     #                 {
#     #                     "type": "reasoning.text",
#     #                     "text": "The user wants a bedtime story about a rabbit. I should use the PromptOptimizerTool to optimize this simple prompt into a better storyline before writing the story. Let me do that first.",
#     #                     "format": "unknown",
#     #                     "index": 0,
#     #                 }
#     #             ],
#     #         ),
#     #         native_finish_reason="tool_calls",
#     #     )
#     # ],
#     created=1784740420,
#     model="nvidia/nemotron-3-ultra-550b-a55b:free",
#     object="chat.completion",
#     moderation=None,
#     service_tier=None,
#     system_fingerprint=None,
#     usage=CompletionUsage(
#         completion_tokens=68,
#         prompt_tokens=347,
#         total_tokens=415,
#         completion_tokens_details=CompletionTokensDetails(
#             accepted_prediction_tokens=None,
#             audio_tokens=0,
#             reasoning_tokens=47,
#             rejected_prediction_tokens=None,
#             image_tokens=0,
#         ),
#         prompt_tokens_details=PromptTokensDetails(
#             audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0
#         ),
#         cost=0,
#         is_byok=False,
#         cost_details={
#             "upstream_inference_cost": 0,
#             "upstream_inference_prompt_cost": 0,
#             "upstream_inference_completions_cost": 0,
#         },
#     ),
#     provider="Nvidia",
# )

In [ ]:
# await openai_chat_completion('hi')